# Baseline (no graph features) — LogReg & Random Forest

Honest tabular floor for the inductive, time-aware Elliptic++ task. **Zero graph-structure features** of any kind: every column is intrinsic to a single transaction and contains no neighborhood / propagated information.

## Features used (108 total)
- 93 `Local_feature_*` — per-tx local properties (BTC value, fees, output volume, etc.)
- 15 named per-tx features:
  - `total_BTC`, `fees`, `size`
  - `num_input_addresses`, `num_output_addresses` (atomic tx structure — every Bitcoin tx has inputs/outputs by definition)
  - `in_BTC_{min,max,mean,median,total}`, `out_BTC_{min,max,mean,median,total}`

## Features explicitly **dropped** (graph-derived)
- 72 `Aggregate_feature_*` — within-timestep 1-hop tx-tx neighborhood aggregates
- `in_txs_degree`, `out_txs_degree` — degrees in the tx-tx graph

No bipartite addr↔tx edges loaded. No wallet trajectory features. No PageRank. This is the strict no-graph baseline.

## Causality / split
- Train: labelled txs with `t ≤ 34`
- Test:  labelled txs with `t ≥ 35`
- Per-tx features are anchored to each tx's own timestep — no leakage by construction (no cross-tx aggregation involved).
- StandardScaler fit on **train only** (matters for LogReg).

## Models
1. **Logistic Regression** (L2, balanced class weight)
2. **Random Forest** (200 trees, balanced class weight) — same hyperparams as Phase 1 RF for direct comparison.

Headline metric: **F1 on the illicit class** (threshold 0.5). Also report AUC and PR-AUC.

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report,
)

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert TRANSACTIONS_DIR.exists(), f"Could not find transactions_data/ from {Path.cwd()}"
print(f"repo root: {ROOT}")

TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
np.random.seed(RANDOM_SEED)

repo root: /Users/bryanfang/Library/CloudStorage/OneDrive-HarvardUniversity/School/2025-2026/STAT 175/175_final_project


In [2]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)

all_cols = list(tx_features_df.columns)
print(f"  total columns: {len(all_cols)}")

# Drop graph-derived columns explicitly
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + [
    "in_txs_degree", "out_txs_degree",
]
META_COLS  = ["txId", "Time step"]

feat_cols = [c for c in all_cols if c not in META_COLS and c not in GRAPH_COLS]
F = len(feat_cols)

n_local = sum(1 for c in feat_cols if c.startswith("Local_feature"))
n_named = F - n_local
print(f"  dropped (graph-derived): {len(GRAPH_COLS)}  "
      f"(72 Aggregate_feature_* + 2 tx-tx degrees)")
print(f"  kept (intrinsic per-tx): {F}  ({n_local} Local_feature_* + {n_named} named)")

# Merge labels
merged = tx_features_df[["txId", "Time step"]].merge(
    tx_classes_df[["txId", "label"]], on="txId", how="left"
)
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
X_all    = tx_features_df[feat_cols].fillna(0).astype(np.float32).values

N = X_all.shape[0]
print(f"  N_tx={N:,}  F={F}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  "
      f"licit={int((tx_label==0).sum()):,}  unknown={int((tx_label==-1).sum()):,}")

Loading transactions...


  total columns: 184
  dropped (graph-derived): 74  (72 Aggregate_feature_* + 2 tx-tx degrees)
  kept (intrinsic per-tx): 108  (93 Local_feature_* + 15 named)
  N_tx=203,769  F=108
  labels: illicit=4,545  licit=42,019  unknown=157,205


In [3]:
labelled   = (tx_label != -1)
train_mask = labelled & (tx_time <= TRAIN_END)
test_mask  = labelled & (tx_time >  TRAIN_END)

X_train_raw = X_all[train_mask]
X_test_raw  = X_all[test_mask]
y_train     = tx_label[train_mask]
y_test      = tx_label[test_mask]
test_t_arr  = tx_time[test_mask]

print(f"train (t<= {TRAIN_END}): n={X_train_raw.shape[0]:,}  illicit_rate={y_train.mean():.4f}")
print(f"test  (t >  {TRAIN_END}): n={X_test_raw.shape[0]:,}  illicit_rate={y_test.mean():.4f}")

# Standardise (fit on TRAIN only — matters for LogReg)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

train (t<= 34): n=29,894  illicit_rate=0.1158
test  (t >  34): n=16,670  illicit_rate=0.0650


In [4]:
print("=== Logistic Regression (no graph features) ===")
t0 = time.time()
lr = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
lr.fit(X_train, y_train)
print(f"  trained in {time.time()-t0:.1f}s")

y_pred_lr  = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]
f1_lr    = f1_score(y_test, y_pred_lr, pos_label=1, zero_division=0)
auc_lr   = roc_auc_score(y_test, y_proba_lr)
prauc_lr = average_precision_score(y_test, y_proba_lr)
print(f"  F1={f1_lr:.4f}  AUC={auc_lr:.4f}  PR-AUC={prauc_lr:.4f}")
print(classification_report(y_test, y_pred_lr,
                            target_names=["licit", "illicit"],
                            digits=4, zero_division=0))

=== Logistic Regression (no graph features) ===


  trained in 2.7s
  F1=0.2430  AUC=0.8686  PR-AUC=0.2634
              precision    recall  f1-score   support

       licit     0.9854    0.6353    0.7725     15587
     illicit     0.1414    0.8643    0.2430      1083

    accuracy                         0.6501     16670
   macro avg     0.5634    0.7498    0.5077     16670
weighted avg     0.9305    0.6501    0.7381     16670



In [5]:
# RF doesn't need scaling — use raw features for fairness with the existing Phase 1 baseline.
print("=== Random Forest (no graph features) ===")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
rf.fit(X_train_raw, y_train)
print(f"  trained in {time.time()-t0:.1f}s")

y_pred_rf  = rf.predict(X_test_raw)
y_proba_rf = rf.predict_proba(X_test_raw)[:, 1]
f1_rf    = f1_score(y_test, y_pred_rf, pos_label=1, zero_division=0)
auc_rf   = roc_auc_score(y_test, y_proba_rf)
prauc_rf = average_precision_score(y_test, y_proba_rf)
print(f"  F1={f1_rf:.4f}  AUC={auc_rf:.4f}  PR-AUC={prauc_rf:.4f}")
print(classification_report(y_test, y_pred_rf,
                            target_names=["licit", "illicit"],
                            digits=4, zero_division=0))

=== Random Forest (no graph features) ===


  trained in 2.8s
  F1=0.8021  AUC=0.9026  PR-AUC=0.7855
              precision    recall  f1-score   support

       licit     0.9792    0.9974    0.9882     15587
     illicit     0.9495    0.6944    0.8021      1083

    accuracy                         0.9777     16670
   macro avg     0.9643    0.8459    0.8952     16670
weighted avg     0.9772    0.9777    0.9761     16670



In [6]:
print("=" * 70)
print("NO-GRAPH BASELINE — summary")
print("=" * 70)
print(f"  {'Model':30s}  {'F1':>8s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("  " + "-" * 60)
for name, f1, auc, prauc in [
    (f"Logistic Regression ({F} feat)", f1_lr, auc_lr, prauc_lr),
    (f"Random Forest        ({F} feat)", f1_rf, auc_rf, prauc_rf),
]:
    print(f"  {name:30s}  {f1:>8.4f}  {auc:>8.4f}  {prauc:>8.4f}")

print("\nReference (from existing notebooks, WITH graph features):")
print(f"  RF[raw 182 incl. Aggregate]    F1=0.8044  AUC=0.9269  PR-AUC=0.7927")
print(f"  RF[raw + 103 traj]             F1=0.8098  AUC=0.9333  PR-AUC=0.8097")
print(f"  RF[raw + traj + 7 L1 PageRank] F1=0.8094  AUC=0.9361  PR-AUC=0.8093")

print("\n" + "=" * 70)
print("Per-timestep test F1 (illicit class)")
print("=" * 70)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  {'LR F1':>8}  {'RF F1':>8}")
for t in range(TRAIN_END + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any():
        continue
    yt = y_test[m]
    if yt.sum() == 0:
        f1_l, f1_r = float("nan"), float("nan")
    else:
        f1_l = f1_score(yt, y_pred_lr[m], pos_label=1, zero_division=0)
        f1_r = f1_score(yt, y_pred_rf[m], pos_label=1, zero_division=0)
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  {f1_l:>8.4f}  {f1_r:>8.4f}")

NO-GRAPH BASELINE — summary
  Model                                 F1       AUC    PR-AUC
  ------------------------------------------------------------
  Logistic Regression (108 feat)    0.2430    0.8686    0.2634
  Random Forest        (108 feat)    0.8021    0.9026    0.7855

Reference (from existing notebooks, WITH graph features):
  RF[raw 182 incl. Aggregate]    F1=0.8044  AUC=0.9269  PR-AUC=0.7927
  RF[raw + 103 traj]             F1=0.8098  AUC=0.9333  PR-AUC=0.8097
  RF[raw + traj + 7 L1 PageRank] F1=0.8094  AUC=0.9361  PR-AUC=0.8093

Per-timestep test F1 (illicit class)
  t      n  illicit     LR F1     RF F1
 35   1341      182    0.4910    0.9607
 36   1708       33    0.0766    0.9143
 37    498       40    0.3022    0.7879
 38    756      111    0.4498    0.9223
 39   1183       81    0.3616    0.9221
 40   1211      112    0.2836    0.7527
 41   1132      116    0.3683    0.9412
 42   2154      239    0.4257    0.8431
 43   1370       24    0.0413    0.0000
 44   1591  

In [7]:
print("Top-20 features by RF importance (no-graph baseline)")
print("=" * 60)
imp = rf.feature_importances_
order = np.argsort(-imp)[:20]
for rank, i in enumerate(order, 1):
    print(f"  {rank:2d}.  {imp[i]:.4f}  {feat_cols[i]}")

print("\nTop-20 features by |LR coefficient| (standardised inputs)")
print("=" * 60)
coef = lr.coef_[0]
order = np.argsort(-np.abs(coef))[:20]
for rank, i in enumerate(order, 1):
    print(f"  {rank:2d}.  {coef[i]:+.4f}  {feat_cols[i]}")

Top-20 features by RF importance (no-graph baseline)
   1.  0.0785  Local_feature_55
   2.  0.0693  Local_feature_53
   3.  0.0561  Local_feature_41
   4.  0.0495  size
   5.  0.0402  Local_feature_49
   6.  0.0400  Local_feature_47
   7.  0.0314  Local_feature_90
   8.  0.0299  Local_feature_43
   9.  0.0266  Local_feature_54
  10.  0.0252  fees
  11.  0.0243  Local_feature_2
  12.  0.0241  Local_feature_14
  13.  0.0240  num_output_addresses
  14.  0.0215  Local_feature_5
  15.  0.0167  Local_feature_59
  16.  0.0159  Local_feature_18
  17.  0.0157  Local_feature_67
  18.  0.0147  Local_feature_52
  19.  0.0143  Local_feature_46
  20.  0.0134  Local_feature_66

Top-20 features by |LR coefficient| (standardised inputs)
   1.  -13.0849  Local_feature_5
   2.  -13.0848  Local_feature_14
   3.  -12.7892  num_output_addresses
   4.  -10.7862  Local_feature_28
   5.  -10.7858  Local_feature_22
   6.  -8.6184  Local_feature_31
   7.  -8.6182  Local_feature_25
   8.  -8.3967  size
   9.  -6.